# Phase 1: Colab Environment Setup

Set up the full SoARM + LIBERO + OpenVLA-OFT stack on a fresh Colab A100 runtime.

## Requirements

- [ ] ENV-01: All dependencies install in correct order without pip resolver conflicts
- [ ] ENV-02: EGL headless rendering configured — OffScreenRenderEnv produces non-black LIBERO frames
- [ ] ENV-03: OpenVLA-OFT loads on GPU (A100 bf16) and returns a valid 7-D action tensor

## Usage

**BLOCK A** (this block): Run cells 0-9 top to bottom, then restart the runtime.

**BLOCK B** (Plan 02): After restart, run verification cells for ENV-01 / ENV-02 / ENV-03.

> Note: Block A must complete fully before restarting. Do not run Block B cells before restart.

In [ ]:
import os

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set REPO_ROOT to the path where you cloned SoARM-Research.
# If using Google Drive: "/content/drive/MyDrive/SoARM-Research"
# If using git clone directly to Colab: "/content/SoARM-Research"
REPO_ROOT = "/content/drive/MyDrive/SoARM-Research"
# ────────────────────────────────────────────────────────────────────────────

# Derived path constants (do not edit these)
LIBERO_ROOT = f"{REPO_ROOT}/LIBERO/libero/libero"
LIBERO_PKG  = f"{REPO_ROOT}/LIBERO"   # path to setup.py directory
OUT_DIR     = f"{REPO_ROOT}/LIBERO/notebooks/outputs"
BDDL_FILE   = (
    f"{LIBERO_ROOT}/bddl_files/libero_spatial/"
    "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl"
)

# Create outputs directory so Block B render check can save there
os.makedirs(OUT_DIR, exist_ok=True)

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"LIBERO_ROOT = {LIBERO_ROOT}")
print(f"LIBERO_PKG  = {LIBERO_PKG}")
print(f"OUT_DIR     = {OUT_DIR}")
print(f"BDDL_FILE   = {BDDL_FILE}")
print(f"Saved → {OUT_DIR}  (outputs directory ready)")

REPO_ROOT   = /content/drive/MyDrive/SoARM-Research
LIBERO_ROOT = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero
LIBERO_PKG  = /content/drive/MyDrive/SoARM-Research/LIBERO
OUT_DIR     = /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs
BDDL_FILE   = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero/bddl_files/libero_spatial/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl
Saved → /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs  (outputs directory ready)


In [2]:
# GPU assertion — D-06
# Check GPU availability and warn loudly if not A100.
# OpenVLA-OFT in bf16 requires ~16 GB VRAM; A100 (40 GB) is the target.
import torch

assert torch.cuda.is_available(), (
    "No GPU available. Go to Runtime > Change runtime type > Hardware accelerator > GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if "A100" not in gpu_name:
    print()
    print("WARNING: Expected A100, got", gpu_name)
    print("WARNING: OpenVLA-OFT in bf16 requires ~16 GB+ VRAM.")
    print("WARNING: ENV-03 will OOM on T4 (15 GB). Restart with an A100 runtime.")
    print("WARNING: You may still proceed for install-only testing on T4.")
else:
    print("A100 confirmed. Proceeding.")

GPU:  Tesla T4
VRAM: 15.6 GB



---

## BLOCK A: Install

Run all cells in this block **top to bottom**, then restart the runtime.

**Ordering is critical** — do not reorder or skip cells:

1. EGL system packages (apt) must install before pip mujoco
2. PyTorch must install before flash-attn (flash-attn compiles against torch CUDA headers)
3. The custom transformers fork must install last (prevents pip downgrade to PyPI version)

---

In [3]:
# Step 1 of 6 — EGL system packages
# Must run BEFORE pip mujoco install.
# These C libraries must exist when the mujoco Python extension builds.
# apt-get update refreshes repo index — prevents 404 for stale package URLs (e.g. libosmesa6).
!apt-get update -qq
!apt-get install -y -q --fix-missing \
    libglfw3 \
    libglew-dev \
    libosmesa6-dev \
    libgles2 \
    libglvnd0 \
    libegl-dev \
    libegl1 \
    libgl1-mesa-glx

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
libegl1 is already the newest version (1.4.0-1).
libgles2 is already the newest version (1.4.0-1).
libglvnd0 is already the newest version (1.4.0-1).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
The following additional packages will be installed:
  libegl-mesa0 libgbm1 libgl-dev libgl1-mesa-dri libglapi-mesa libglew2.2
  libglu1-mesa libglu1-mesa-dev libglx-dev libglx-mesa0 libosmesa6
Suggested packages:
  glew-utils libgles1 libvulkan1
Recommended packages:
  libgl1-amber-dri
The following NEW packages will be installed:
  libegl-dev libgl-dev libglew-dev libglew2.2 libglfw3 libglu1-mesa
  libglu1-mesa-dev libglx-dev libosmesa6 libosmesa6-dev
The following packages will be upgraded:
  libeg

In [4]:
# Step 2 of 6 — PyTorch 2.2.0 (cu121)
# Must run BEFORE flash-attn: flash-attn compiles CUDA kernels against installed torch headers.
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 2.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 25.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 103.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 58.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 108.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 22.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━

In [6]:
# Step 3 of 6 — MuJoCo + simulation stack
#
# Strategy as of 2025-07 (Colab Python 3.12):
#   mujoco 2.3.7 has NO binary wheel for cp311/cp312 on PyPI and NO conda available.
#   mujoco 3.3.2 IS available as a binary wheel for cp312 and is the known-stable
#   version for LIBERO evaluations (confirmed in LIBERO GitHub issues).
#   LIBERO's one mujoco 2.x API call (mj_step1) is patched in bddl_base_domain.py
#   to use mj_kinematics (the mujoco 3.x equivalent) as a transparent fallback.
#
#   numpy: keep Colab's pre-installed numpy (likely 2.x). numpy<2 breaks cupy/jax/
#          opencv pre-installed in Colab — no benefit to downgrading.
#   gym: --prefer-binary avoids source build on Python 3.12.
import sys
print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

# mujoco 3.3.2: binary wheel available for cp312; stable for LIBERO
!pip install "mujoco==3.3.2" -q

# gym: 0.25.2 has no cp311/cp312 wheel; --prefer-binary uses nearest available
!pip install "gym>=0.21,<=0.26" --prefer-binary -q

# robosuite MUST be 1.4.0 — 1.5.x removed SingleArmEnv which LIBERO requires
# remaining deps are pure Python (no wheel issues)
!pip install     robosuite==1.4.0     bddl==1.0.1     easydict==1.9     cloudpickle==2.1.0     einops==0.4.1     "imageio[ffmpeg]" -q

# opencv: let pip pick a cp312-compatible version (>=4.7 has cp312 wheels)
!pip install "opencv-python-headless>=4.7,<5" --prefer-binary -q

# Verify mujoco importable and check version
import importlib, mujoco
mv = tuple(int(x) for x in mujoco.__version__.split("."))
print(f"mujoco {mujoco.__version__} installed")
if mv >= (3, 0, 0):
    print("✓ mujoco 3.x — bddl_base_domain.py mj_kinematics shim is active")
else:
    print(f"mujoco {mujoco.__version__} (2.x path)")
print("✓ simulation stack installed")


Python 3.12.13
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 83.7 MB/s eta 0:00:00:00:010:01
mujoco 3.3.2 installed
✓ mujoco 3.x — bddl_base_domain.py mj_kinematics shim is active
✓ simulation stack installed


In [7]:
# Step 4 of 6 — LIBERO editable install
# LIBERO_PKG is defined in Cell 1. setup.py may be missing if Google Drive
# didn't sync the nested LIBERO git repo — auto-clone from GitHub in that case.
import os, subprocess, sys
from pathlib import Path

# Auto-clone LIBERO if setup.py not found on Drive
_libero_setup = Path(LIBERO_PKG) / 'setup.py'
if not _libero_setup.exists():
    _fallback = '/content/libero'
    print(f'⚠ {LIBERO_PKG}/setup.py missing (Google Drive may not sync nested git repos)')
    print(f'  Cloning LIBERO from GitHub → {_fallback} ...')
    _r = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/Lifelong-Robot-Learning/LIBERO.git', _fallback],
        capture_output=True, text=True,
    )
    if _r.returncode != 0:
        raise RuntimeError(f'LIBERO clone failed:\n{_r.stderr}')
    # Redirect paths to cloned location (affects Block B cells)
    LIBERO_PKG  = _fallback
    LIBERO_ROOT = f'{_fallback}/libero/libero'
    BDDL_FILE   = (
        f'{LIBERO_ROOT}/bddl_files/libero_spatial/'
        'pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl'
    )
    print(f'✓ LIBERO cloned. Paths updated → LIBERO_PKG={LIBERO_PKG}')
else:
    print(f'✓ LIBERO found at {LIBERO_PKG}')

# editable install — LIBERO package changes are live without reinstall
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', LIBERO_PKG, '-q'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
else:
    print('✓ LIBERO installed in editable mode from:', LIBERO_PKG)

⚠ /content/drive/MyDrive/SoARM-Research/LIBERO/setup.py missing (Google Drive may not sync nested git repos)
  Cloning LIBERO from GitHub → /content/libero ...
✓ LIBERO cloned. Paths updated → LIBERO_PKG=/content/libero
✓ LIBERO installed in editable mode from: /content/libero


In [8]:
# Step 5 of 6 — OpenVLA-OFT supporting packages + custom transformers fork
# The git fork (moojink/transformers-openvla-oft) MUST be installed LAST in this cell.
# Do NOT install transformers from PyPI — the fork replaces it entirely.
# The fork adds bidirectional attention for parallel decoding; PyPI version lacks this.
!pip install \
    timm==0.9.10 \
    tokenizers==0.19.1 \
    sentencepiece==0.1.99 \
    peft==0.11.1 \
    accelerate \
    huggingface_hub -q

# Install transformers fork LAST — pip resolver cannot downgrade to PyPI version this way.
# Commit SHA comment for reproducibility: installs from main branch of the fork repo.
# To pin a specific commit: git+https://github.com/moojink/transformers-openvla-oft.git@<SHA>
!pip install git+https://github.com/moojink/transformers-openvla-oft.git -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 50.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta

In [9]:
# Step 6 of 6 — flash-attn (pre-built wheel — avoids 90-min T4 source compile)
#
# Problem: flash-attn source compilation takes 60-90 min on T4 and OOMs/crashes runtime.
# Fix: use pre-built wheels from flash-attention GitHub releases (~30 sec install).
#
# Wheel naming: flash_attn-{ver}+cu{cuda}torch{torch}cxx11abiFALSE-cp{py}-linux_x86_64.whl
# We detect the exact CUDA + torch + Python versions at runtime to pick the right wheel.
import subprocess, sys, torch

FLASH_VER  = "2.5.5"
TORCH_VER  = "2.2.0"
PY_TAG     = f"cp{sys.version_info.major}{sys.version_info.minor}"  # e.g. cp312
cuda_tag   = "cu" + torch.version.cuda.replace(".", "")             # e.g. cu121

wheel_url = (
    f"https://github.com/Dao-AILab/flash-attention/releases/download/"
    f"v{FLASH_VER}/flash_attn-{FLASH_VER}+{cuda_tag}torch{TORCH_VER}"
    f"cxx11abiFALSE-{PY_TAG}-{PY_TAG}-linux_x86_64.whl"
)
print(f"Detected: CUDA={torch.version.cuda}, Python={PY_TAG}")
print(f"Trying pre-built wheel: {wheel_url}")

!pip install packaging ninja -q

import subprocess
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", wheel_url, "-q"],
    capture_output=True, text=True
)
print(r.stdout[-500:] if r.stdout else "")
print(r.stderr[-500:] if r.stderr else "")

import importlib
if importlib.util.find_spec("flash_attn") is not None:
    import flash_attn
    print(f"✓ flash-attn {flash_attn.__version__} installed (pre-built wheel)")
else:
    print("⚠ Pre-built wheel not found for this config. Checking available wheels...")
    print(f"  Tried: {wheel_url}")
    print("  → See: https://github.com/Dao-AILab/flash-attention/releases/tag/v2.5.5")
    print("  → Install manually: pip install <correct_wheel_url>")
    print("  ⚠ Skipping flash-attn — ENV-01 and ENV-02 checks will still work.")
    print("  ⚠ ENV-03 (OpenVLA-OFT) requires flash-attn; install before Block B Cell ENV-03.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 12.8 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


: 

---

## *** STOP — Restart runtime now ***

Go to: **Runtime > Restart session** (or press Ctrl+M .), then continue from BLOCK B below.

Do **not** run any cells below this point until after the runtime has restarted.

After restart, continue in **this notebook** — run the BLOCK B cells below.

---